# Exploratory Data Analysis — WTI Crude Oil
Exploring the WTI crude oil daily price data used for model training.  
Data source: Yahoo Finance (`CL=F`)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yfinance as yf
import warnings
warnings.filterwarnings("ignore")

BG, CARD  = "#0f1117", "#1c1e26"
BLUE, GREEN, AMBER, RED = "#378ADD", "#1D9E75", "#EF9F27", "#E24B4A"
TEXT, MUTED, GRID = "#c9d1d9", "#6e7681", "#30363d"

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor":   CARD,
    "axes.edgecolor":   GRID,
    "axes.labelcolor":  MUTED,
    "xtick.color":      MUTED,
    "ytick.color":      MUTED,
    "text.color":       TEXT,
    "grid.color":       GRID,
    "grid.linestyle":   "--",
    "grid.linewidth":   0.5,
})

print("Libraries loaded.")

## 1. Fetch data

In [ ]:
df = yf.download("CL=F", start="2000-01-01", end="2026-04-01", interval="1d", progress=False)
df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
df = df[["Open","High","Low","Close","Volume"]].dropna()
df.index = pd.to_datetime(df.index)
df = df.asfreq("B").ffill()

print(f"Shape     : {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
print(f"\nFirst 5 rows:")
df.head()

## 2. Basic statistics

In [ ]:
print("Descriptive statistics:")
df.describe().round(2)

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

## 3. Price history

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
fig.patch.set_facecolor(BG)

# Price
axes[0].plot(df.index, df["Close"], color=BLUE, lw=1.2, label="Close")
axes[0].fill_between(df.index, df["Low"], df["High"], alpha=0.15, color=BLUE)
axes[0].set_ylabel("Price ($/bbl)")
axes[0].set_title("WTI Crude Oil — Daily closing price 2000–2026", color=TEXT)
axes[0].legend(fontsize=9)

# Key events
events = {
    "2008\nCrisis":    "2008-09-15",
    "COVID\n2020":     "2020-03-20",
    "Ukraine\n2022":   "2022-02-24",
    "OPEC+\n2023":     "2023-06-04",
}
for label, date in events.items():
    axes[0].axvline(pd.Timestamp(date), color=RED, lw=1, linestyle="--", alpha=0.7)
    axes[0].text(pd.Timestamp(date), df["Close"].max() * 0.85,
                 label, color=RED, fontsize=8, ha="center")

# Volume
axes[1].bar(df.index, df["Volume"] / 1e6, color=AMBER, alpha=0.7, width=1)
axes[1].set_ylabel("Volume (M)")
axes[1].set_xlabel("Date")

plt.tight_layout()
plt.savefig("eda_price_history.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()

## 4. Feature engineering

In [ ]:
df["Change_pct"] = df["Close"].pct_change() * 100
df["MA4"]        = df["Close"].rolling(4).mean()
df["MA12"]       = df["Close"].rolling(12).mean()
df["Momentum"]   = df["Close"].diff()
df["Volatility"] = df["Close"].rolling(4).std()
df["HL_Spread"]  = df["High"] - df["Low"]
df = df.dropna()

print(f"Shape after feature engineering: {df.shape}")
print(f"\nFeatures: {list(df.columns)}")
df.head()

## 5. Feature distributions

In [ ]:
features = ["Close","Change_pct","MA4","MA12","Momentum","Volatility","HL_Spread"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.patch.set_facecolor(BG)
axes = axes.flatten()

colors = [BLUE, AMBER, GREEN, RED, BLUE, AMBER, GREEN]
for i, (feat, color) in enumerate(zip(features, colors)):
    axes[i].hist(df[feat], bins=60, color=color, alpha=0.8, edgecolor="none")
    axes[i].set_title(feat, color=TEXT, fontsize=11)
    axes[i].set_ylabel("Count")

axes[-1].axis("off")
plt.suptitle("Feature distributions", color=TEXT, fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("eda_distributions.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()

## 6. Correlation matrix

In [ ]:
import matplotlib.colors as mcolors

corr = df[features].corr()

fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor(BG)
ax.set_facecolor(CARD)

cmap = plt.cm.RdYlGn
im = ax.imshow(corr, cmap=cmap, vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(features)))
ax.set_yticks(range(len(features)))
ax.set_xticklabels(features, rotation=45, ha="right", fontsize=10)
ax.set_yticklabels(features, fontsize=10)

for i in range(len(features)):
    for j in range(len(features)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}",
                ha="center", va="center", fontsize=8,
                color="black" if abs(corr.iloc[i,j]) < 0.7 else "white")

ax.set_title("Feature correlation matrix", color=TEXT, fontsize=13)
plt.tight_layout()
plt.savefig("eda_correlation.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()

## 7. Price delta analysis
Understanding the prediction target used by all 3 models.

In [ ]:
deltas = df["Close"].diff().dropna()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor(BG)

# Delta over time
axes[0].plot(deltas.index, deltas, color=BLUE, lw=0.8, alpha=0.7)
axes[0].axhline(0, color=MUTED, lw=1)
axes[0].set_title("Daily price delta over time", color=TEXT)
axes[0].set_ylabel("Delta ($/bbl)")

# Delta distribution
axes[1].hist(deltas, bins=100, color=AMBER, alpha=0.8, edgecolor="none")
axes[1].axvline(deltas.mean(), color=RED, lw=1.5, label=f"Mean: {deltas.mean():.3f}")
axes[1].axvline(deltas.std(), color=GREEN, lw=1.5, linestyle="--", label=f"Std: {deltas.std():.3f}")
axes[1].set_title("Delta distribution", color=TEXT)
axes[1].set_xlabel("Delta ($/bbl)")
axes[1].legend(fontsize=9)

# Delta stats
stats = {
    "Mean":     f"{deltas.mean():.4f}",
    "Std":      f"{deltas.std():.4f}",
    "Min":      f"{deltas.min():.2f}",
    "Max":      f"{deltas.max():.2f}",
    "Skewness": f"{deltas.skew():.4f}",
    "Kurtosis": f"{deltas.kurtosis():.4f}",
}
axes[2].axis("off")
y = 0.9
for k, v in stats.items():
    axes[2].text(0.1, y, k, transform=axes[2].transAxes,
                 color=MUTED, fontsize=12)
    axes[2].text(0.6, y, v, transform=axes[2].transAxes,
                 color=TEXT, fontsize=12, fontweight="bold")
    y -= 0.14
axes[2].set_title("Delta statistics", color=TEXT)

plt.tight_layout()
plt.savefig("eda_delta_analysis.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()

## 8. Volatility regimes
Identifying high and low volatility periods — important for model evaluation.

In [ ]:
rolling_vol = df["Close"].pct_change().rolling(30).std() * np.sqrt(252) * 100

fig, ax = plt.subplots(figsize=(15, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(CARD)

ax.plot(rolling_vol.index, rolling_vol, color=AMBER, lw=1.2)
ax.fill_between(rolling_vol.index, rolling_vol,
                where=rolling_vol > rolling_vol.quantile(0.75),
                color=RED, alpha=0.3, label="High volatility")
ax.fill_between(rolling_vol.index, rolling_vol,
                where=rolling_vol < rolling_vol.quantile(0.25),
                color=GREEN, alpha=0.3, label="Low volatility")
ax.set_title("30-day rolling annualised volatility", color=TEXT, fontsize=13)
ax.set_ylabel("Volatility (%)")
ax.legend(fontsize=10)
ax.grid(True)

plt.tight_layout()
plt.savefig("eda_volatility.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()

print(f"Mean volatility : {rolling_vol.mean():.1f}%")
print(f"Max volatility  : {rolling_vol.max():.1f}%")
print(f"Min volatility  : {rolling_vol.min():.1f}%")